In [1]:
!pip install fastapi uvicorn supabase python-dotenv


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 3.8 MB/s eta 0:00:00


In [2]:
from fastapi import FastAPI
import uvicorn

app = FastAPI()

@app.get("/")
def home():
    return {
        "status": "Mela al mando",
        "mensaje": "Backend de Andes City funcionando en la nube"
    }

print("Servidor configurado. Listo para despegar.")


Servidor configurado. Listo para despegar.


In [4]:
# Este código sirve para ejecutar el servidor dentro de Colab
import nest_asyncio
from pyngrok import ngrok

# Permitir que el servidor corra en este ambiente
nest_asyncio.apply()

# Iniciar el servidor de prueba
# Nota: En Colab no podemos usar el comando de terminal directo fácilmente,
# así que usamos esta línea de Python:
if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)

ModuleNotFoundError: No module named 'pyngrok'

In [5]:
!pip install pyngrok nest-asyncio

In [7]:
from fastapi import FastAPI
import uvicorn
import nest_asyncio
from pyngrok import ngrok

app = FastAPI()

# Esta es la ruta que te pidió el equipo (Endpoint de prueba)
@app.get("/")
def read_root():
    return {
        "status": "Mela al mando",
        "proyecto": "Andes City",
        "mensaje": "¡Backend funcionando desde la nube!"
    }

# Aplicar el parche para que funcione en Colab
nest_asyncio.apply()

# Abrir el túnel público
public_url = ngrok.connect(8000).public_url
print(f"🚀 TU BACKEND ESTÁ VIVO EN ESTA URL: {public_url}")


🚀 TU BACKEND ESTÁ VIVO EN ESTA URL: https://fleshlier-juxtapositional-latosha.ngrok-free.dev


In [8]:
if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)

RuntimeError: asyncio.run() cannot be called from a running event loop

In [9]:
import asyncio

# Configuración especial para que el servidor no choque con Colab
if __name__ == "__main__":
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
    server = uvicorn.Server(config)

    # En lugar de uvicorn.run, usamos este truco para Colab:
    loop = asyncio.get_event_loop()
    loop.create_task(server.serve())

    print(f"✅ SERVIDOR ACTIVO EN SEGUNDO PLANO")
    print(f"🔗 Tu URL pública sigue siendo: {public_url}")

✅ SERVIDOR ACTIVO EN SEGUNDO PLANO
🔗 Tu URL pública sigue siendo: https://fleshlier-juxtapositional-latosha.ngrok-free.dev


In [11]:
# Traer todas las filas para ver qué agregó Dennys
try:
    check_weather = supabase.table("weather_log").select("*").execute()
    if check_weather.data:
        print("✅ ¡Datos encontrados en Clima!")
        print(f"Columnas detectadas: {list(check_weather.data[0].keys())}")
        print(f"Total de filas: {len(check_weather.data)}")
        # Mostrar la primera fila para revisar los datos
        print("Ejemplo de fila:", check_weather.data[0])
    else:
        print("⚠️ La tabla 'weather_log' existe pero está vacía.")
except Exception as e:
    print(f"❌ Error al conectar: {e}")

✅ ¡Datos encontrados en Clima!
Columnas detectadas: ['id', 'temp_c', 'condition', 'alert_level', 'recorded_at']
Total de filas: 2
Ejemplo de fila: {'id': 1, 'temp_c': 10.5, 'condition': 'Lluvia Fuerte', 'alert_level': 'precaucion', 'recorded_at': '2026-03-29T21:54:43.400634+00:00'}


In [13]:
@app.get("/api/weather/current")
def get_weather():
    response = supabase.table("weather_log").select("*").order("recorded_at", desc=True).limit(1).execute()
    return response.data[0] if response.data else {"error": "Sin datos"}

@app.get("/api/routes/status")
def get_routes():
    # Cambiamos 'street_name' por 'name' que es como lo puso Dennys
    response = supabase.table("mobility_routes").select("name, status, travel_time_min").execute()
    return {"routes": response.data} if response.data else {"error": "Sin rutas"}

In [15]:
@app.get("/api/weather/current")
def get_weather():
    response = supabase.table("weather_log").select("*").order("recorded_at", desc=True).limit(1).execute()
    return response.data[0] if response.data else {"error": "Sin datos"}

@app.get("/api/routes/status")
def get_routes():
    # Cambiamos 'street_name' por 'name' que es como lo puso Dennys
    response = supabase.table("mobility_routes").select("name, status, travel_time_min").execute()
    return {"routes": response.data} if response.data else {"error": "Sin rutas"}

In [12]:
try:
    check_routes = supabase.table("mobility_routes").select("*").execute()
    if check_routes.data:
        print("✅ ¡Datos encontrados en Rutas!")
        print(f"Columnas detectadas: {list(check_routes.data[0].keys())}")
        print(f"Ejemplo de calle: {check_routes.data[0].get('street_name', 'N/A')}")
    else:
        print("⚠️ La tabla 'mobility_routes' está vacía.")
except Exception as e:
    print(f"❌ Error al conectar: {e}")


✅ ¡Datos encontrados en Rutas!
Columnas detectadas: ['id', 'name', 'status', 'affected_by_weather', 'travel_time_min', 'updated_at']
Ejemplo de calle: N/A


In [21]:
try:
    # Intentamos pedir una sola cosa a la tabla de clima
    supabase.table("weather_log").select("id").limit(1).execute()
    print("🟢 CONEXIÓN ACTIVA: El túnel hacia la base de datos está abierto.")
except Exception as e:
    print("🔴 ERROR DE CONEXIÓN: Revisa si las llaves o la URL son correctas.")

🟢 CONEXIÓN ACTIVA: El túnel hacia la base de datos está abierto.


In [22]:
from google.colab import userdata
from supabase import create_client, Client

# Ya no escribimos las llaves aquí, las traemos de los secretos de Colab
url = userdata.get('SUPABASE_URL')
key = userdata.get('SUPABASE_KEY')

# Conexión segura
supabase: Client = create_client(url, key)

print("✅ Conexión establecida de forma segura (Llaves ocultas)")

✅ Conexión establecida de forma segura (Llaves ocultas)


In [23]:
import sys
print(sys.version)

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [25]:
from google.colab import userdata
import os

# Así se llama al token de forma segura
token = userdata.get('NGROK_TOKEN')
os.system(f"ngrok config add-authtoken {token}")

0

In [27]:
@app.get("/api/user/profile/{user_id}")
def get_user_profile(user_id: str):
    # Consultamos la tabla 'users' que es la pública según el esquema
    response = supabase.table("users").select("username, role, verified, avatar_url").eq("id", user_id).single().execute()

    if response.data:
        return response.data

    return {"error": "Perfil no encontrado"}

In [28]:
from fastapi import Request, HTTPException

@app.middleware("http")
async def auth_middleware(request: Request, call_next):
    # 1. Permitimos que el Login y el Clima sean públicos para que la app no se bloquee al abrirse
    public_paths = ["/api/auth/login", "/api/weather/current", "/docs", "/openapi.json"]
    if request.url.path in public_paths:
        return await call_next(request)

    # 2. Revisamos si Alfonso envía el ID del usuario
    user_id = request.headers.get("X-User-ID") # Alfonso debe configurar esto en el Frontend

    if not user_id:
        # Aquí es donde "rebotamos" al usuario
        raise HTTPException(status_code=401, detail="Acceso denegado: Inicia sesión primero")

    return await call_next(request)

RuntimeError: Cannot add middleware after an application has started

In [2]:
from fastapi import FastAPI, Request, HTTPException
from supabase import create_client, Client
from google.colab import userdata
import nest_asyncio
import uvicorn

# 1. Cargar credenciales desde Secrets
url = userdata.get('SUPABASE_URL')
key = userdata.get('SUPABASE_KEY')
supabase: Client = create_client(url, key)

# 2. Crear la aplicación
app = FastAPI()

In [9]:
from functools import wraps
from flask import request, jsonify

def auth_middleware(f):
    @wraps(f)
    def decorated(*args, **kwargs):
        # El frontend de Alfonso debe enviar el token en el header 'Authorization'
        token = request.headers.get('Authorization')

        if not token:
            return jsonify({"error": "Acceso denegado. Token no encontrado"}), 401

        try:
            # Verificación del token con Supabase
            user = supabase.auth.get_user(token)
            if user:
                return f(user, *args, **kwargs)
        except Exception as e:
            return jsonify({"error": "Sesión inválida o expirada"}), 401

    return decorated

/usr/local/lib/python3.12/dist-packages/werkzeug/datastructures/file_storage.py:12: RuntimeWarning: coroutine 'Server.serve' was never awaited
  from .headers import Headers


In [13]:
@app.route('/profile', methods=['GET'])
@auth_middleware
def get_profile(user_data):
    try:
        # El user_id se extrae del token validado por el middleware
        user_id = user_data.user.id

        # Consultamos la tabla 'users' que se muestra en tu esquema
        # Nota: Seleccionamos los campos exactos de tu imagen
        response = supabase.table('users') \
            .select('username, role, verified, avatar_url') \
            .eq('id', user_id) \
            .single() \
            .execute()

        return jsonify(response.data), 200

    except Exception as e:
        return jsonify({"error": "Usuario no encontrado en la tabla users", "details": str(e)}), 404

In [14]:
# 1. Instalación (ejecutar una sola vez)
# !pip install supabase flask_cors

from supabase import create_client, Client
from google.colab import userdata # Import userdata

# 2. Configuración
# Recuperar URL y KEY de forma segura desde los secretos de Colab
url = userdata.get('SUPABASE_URL')
key = userdata.get('SUPABASE_KEY')
supabase: Client = create_client(url, key)

# 3. El Middleware (Puente)
def auth_middleware(f):
    def wrapper(*args, **kwargs):
        # Alfonso debe enviar el token aquí
        token = request.headers.get("Authorization")
        if not token:
            return jsonify({"msg": "Token faltante"}), 401

        # Verificación directa con el servidor de Auth de Supabase
        user = supabase.auth.get_user(token.replace("Bearer ", ""))
        if user:
            return f(user, *args, **kwargs)
        return jsonify({"msg": "Token inválido"}), 401
    return wrapper

SupabaseException: Invalid URL

In [15]:
from google.colab import userdata
from supabase import create_client, Client

# 2. Configuración usando Secretos de Colab
try:
    url = userdata.get('SUPABASE_URL')
    key = userdata.get('SUPABASE_KEY')
    supabase: Client = create_client(url, key)
    print("Conexión exitosa con el puente de Supabase")
except Exception as e:
    print(f"Error al conectar: {e}")

# 3. El Middleware (Puente) - Continúa con tu código...

Conexión exitosa con el puente de Supabase


In [17]:
def verificar_sesion_activa(token_recibido):
    try:
        # 1. Validamos el token con el servicio de Auth
        user_auth = supabase.auth.get_user(token_recibido)
        uid = user_auth.user.id

        # 2. Si el token es válido, buscamos sus datos extendidos en tu tabla 'users'
        # Según tu esquema: id, username, role, verified
        perfil = supabase.table('users').select('username, role, verified').eq('id', uid).single().execute()

        if perfil.data:
            print(f"Acceso Concedido: Bienvenido {perfil.data['username']} ({perfil.data['role']})")
            return perfil.data
        else:
            print("Error: El usuario existe en Auth pero no en tu tabla 'users'.")
            return None

    except Exception as e:
        print(f"Acceso Denegado: Token inválido o expirado. {e}")
        return None

In [18]:
# access_point.py (o el nombre que prefieras)
from supabase import create_client, Client
from google.colab import userdata # Import userdata

# Las credenciales se leen de los secretos de Colab, no se escriben aquí
url = userdata.get('SUPABASE_URL')
key = userdata.get('SUPABASE_KEY')
supabase: Client = create_client(url, key)

def auth_middleware(token):
    # Lógica de validación que definimos...
    pass

SupabaseException: supabase_url is required

In [19]:
import os
from supabase import create_client, Client

# Intentar obtener de Google Colab Secrets primero
try:
    from google.colab import userdata
    url = userdata.get('SUPABASE_URL')
    key = userdata.get('SUPABASE_KEY')
except ImportError:
    # Si no es Colab, buscar en variables de entorno locales (.env)
    url = os.environ.get('SUPABASE_URL')
    key = os.environ.get('SUPABASE_KEY')

# Validación de seguridad antes de inicializar
if not url or not key:
    raise ValueError("Error: SUPABASE_URL o SUPABASE_KEY no configuradas.")

supabase: Client = create_client(url, key)
print("Conexión establecida correctamente.")

Conexión establecida correctamente.


In [23]:
def middleware_mela(token_de_alfonso):
    try:
        # El puente le pregunta a Supabase: "¿Este token es real?"
        user = supabase.auth.get_user(token_de_alfonso)
        return user.user.id # Si es real, nos da su ID único
    except:
        return None # Si es falso, devuelve nada (acceso denegado)

In [24]:
def middleware_mela(token_de_alfonso):
    try:
        print("Verificando token con Supabase...")

        # El puente le pregunta a Supabase: "¿Este token es real?"
        user_response = supabase.auth.get_user(token_de_alfonso)

        # Si llegamos aquí, el token es válido
        user_id = user_response.user.id
        print(f"Token válido. UUID detectado: {user_id}")
        return user_id

    except Exception as e:
        # Si el token es inventado o expiró, saltará aquí
        print(f"Acceso Denegado: {e}")
        return None

In [25]:
# Prueba 1: Con un token falso (debería imprimir "Acceso Denegado")
resultado = middleware_mela("token_inventado_123")
print(f"Resultado final: {resultado}")

Verificando token con Supabase...
Acceso Denegado: invalid JWT: unable to parse or verify signature, token is malformed: token contains an invalid number of segments
Resultado final: None


In [27]:
def access_point_completo(token_recibido):
    # 1. El Middleware revisa la llave
    uid_detectado = middleware_mela(token_recibido)

    if uid_detectado:
        print(f"Éxito: Buscando datos para el UUID: {uid_detectado}")

        # 2. Si es válido, buscamos en tu tabla 'users' del esquema
        perfil = supabase.table('users').select('username, role, ciudad').eq('id', uid_detectado).single().execute()

        return perfil.data
    else:
        return {"error": "No tienes permiso para estar aquí. Inicia sesión."}

# Nota: Cuando Dennys te dé el UUID, lo usaremos para saltarnos el paso del token
# y probar la base de datos directamente.

In [28]:
def get_user_profile(token_from_alfonso):
    """
    Esta es la función principal del Access Point de Mela.
    Recibe el token, lo valida y devuelve los datos de la tabla 'users'.
    """
    # 1. El Middleware verifica si el usuario es real
    user_id = middleware_mela(token_from_alfonso)

    if not user_id:
        return {"error": "No autorizado", "status": 401}

    # 2. Si es real, extraemos su información de tu tabla 'users'
    try:
        perfil = supabase.table('users') \
            .select('username, role, verified, avatar_url') \
            .eq('id', user_id) \
            .single() \
            .execute()

        print(f"Perfil de {perfil.data['username']} enviado al frontend.")
        return perfil.data

    except Exception as e:
        return {"error": "Usuario no encontrado en la tabla users", "details": str(e)}